# VinDr-PCXR Download Script
This notebook downloads metadata and DICOM images from PhysioNet. It handles authentication, automated metadata retrieval, and skips existing files.

In [3]:
import os
import sys
import yaml
import pandas as pd
import requests
from tqdm.auto import tqdm
from pathlib import Path

# --- Configuration ---
SELECT_SPLIT = 'test' # 'train' or 'test'

# 1. Find project root and load config
PROJECT_ROOT = Path.cwd().parent
CONFIG_PATH = Path.cwd() / "config.yml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

# 2. Determine base path from config relative to root
BASE_PATH = PROJECT_ROOT / 'machex' / config["VINDR_PCXR_ROOT"]

# 3. Determine paths based on split
LABEL_CSV = BASE_PATH / SELECT_SPLIT / f"image_labels_{SELECT_SPLIT}.csv"
OUTPUT_DIR = BASE_PATH / SELECT_SPLIT
LIST_FILE = BASE_PATH / f"download_list_{SELECT_SPLIT}.txt"
BASE_URL = f"https://physionet.org/files/vindr-pcxr/1.0.0/{SELECT_SPLIT}/"

print(f"Mode: {SELECT_SPLIT}")
print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Mode: test
Project root: D:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl
Output directory: D:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\machex\chestx-ray_test\VinDr-PCXR\test


## 1. Login and Download Metadata CSVs
This section authenticates and ensures you have the label files before proceeding.

In [4]:
session = requests.Session()

print("Logging in to PhysioNet...")
login_url = "https://physionet.org/login/"
get_response = session.get(login_url)
csrftoken = session.cookies.get('csrftoken')

login_data = {
    'username': config["PHYSIO_USERNAME"],
    'password': config["PHYSIO_PASSWORD"],
    'csrfmiddlewaretoken': csrftoken,
    'next': '/content/vindr-pcxr/1.0.0/'
}
session.post(login_url, data=login_data, headers={'Referer': login_url})

def download_metadata(split):
    # Files to download for each split
    targets = [
        (f"image_labels_{split}.csv", f"https://physionet.org/files/vindr-pcxr/1.0.0/image_labels_{split}.csv"),
        (f"annotations_{split}.csv", f"https://physionet.org/files/vindr-pcxr/1.0.0/annotations_{split}.csv")
    ]
    
    for filename, url in targets:
        dest = BASE_PATH / split / filename
        if dest.exists():
            print(f"  ✓ {filename} already exists.")
            continue
        
        dest.parent.mkdir(parents=True, exist_ok=True)
        print(f"  Downloading {filename}...")
        
        res = session.get(url)
        if res.status_code == 200:
            with open(dest, 'wb') as f:
                f.write(res.content)
            print(f"  ✓ Saved to {dest.parent}")
        else:
            print(f"  ✖ Failed to download {filename} (Status {res.status_code})")

print("Checking metadata files:")
download_metadata('train')
download_metadata('test')

Logging in to PhysioNet...
Checking metadata files:
  ✓ image_labels_train.csv already exists.
  ✓ annotations_train.csv already exists.
  ✓ image_labels_test.csv already exists.
  ✓ annotations_test.csv already exists.


## 2. Load Metadata and Create Download List

In [5]:
if not LABEL_CSV.exists():
    raise FileNotFoundError(f"Metadata CSV not found at {LABEL_CSV}")

df = pd.read_csv(LABEL_CSV)
print(f"Loaded {len(df)} image IDs from {LABEL_CSV.name}")

with open(LIST_FILE, 'w') as f:
    for img_id in df['image_id']:
        f.write(f"{BASE_URL}{img_id}.dicom\n")

print(f"Created download list at {LIST_FILE.name} with {len(df)} URLs.")

Loaded 1397 image IDs from image_labels_test.csv
Created download list at download_list_test.txt with 1397 URLs.


## 3. Execute Authenticated Download

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(LIST_FILE, 'r') as f:
    urls = [line.strip() for line in f if line.strip()]

print(f"Starting download of {len(urls)} files...")

for url in tqdm(urls):
    filename = OUTPUT_DIR / os.path.basename(url)
    
    if filename.exists() and filename.stat().st_size > 0:
        continue

    response = session.get(url, stream=True)
    
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024): # 1MB Chunks
                f.write(chunk)
    elif response.status_code == 403:
        print(f"\nError 403 at {url}. Check login or Data Use Agreement (DUA).")
        break 
    else:
        print(f"\nError {response.status_code} at {url}")

print("\nDownload process complete!")

Starting download of 1397 files...


  0%|          | 0/1397 [00:00<?, ?it/s]

  0%|          | 1/1397 [00:20<7:57:29, 20.52s/it]


KeyboardInterrupt: 